# HMM specialist diffusion

Same MVO tests as `Notebook_Data_5`, but synthetic series come from **HMM-path specialist pools**, not calendar date-matched panels.

Real training data is the last 252 days. Synthetic columns are stitched along a causal HMM path and mixed with `mix_train_with_regime_paths` (no date lookup). Metrics: Sharpe, Calmar, max drawdown, CVaR.

Open with working directory `evaluation/`.


## Setup


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

EVAL_DIR = Path.cwd()
if not (EVAL_DIR / "specialist_eval.py").exists():
    EVAL_DIR = Path.cwd() / "evaluation"
PROJECT_ROOT = EVAL_DIR.parent
sys.path.insert(0, str(EVAL_DIR))

from data_utils import load_real_data
from portfolio_core import pct_returns
from portfolio_eval_final import plot_metric_curve
from specialist_eval import (
    ASSET_COLS,
    daily_paths,
    load_regime_pools,
    paired_deltas,
    path_metrics,
    rolling_experiment,
    shock_experiment,
    specialist_corr_matrix,
    static_paths,
)

SEED = 42
SEEDS = [42, 43, 1034, 45, 88]
LOOKBACK, INVEST, STEP = 252, 60, 21
MIX_GRID = (0, 1, 2, 5, 10, 15, 20)
N_SYNTH_LIST = [0, 1, 2, 5, 20]
MAX_W = 0.25
HMM_STEP = 21
np.random.seed(SEED)

prices = load_real_data(
    PROJECT_ROOT / "evaluation" / "data" / "benchmark" / "benchmark_data.csv",
    test_start_year="2013",
    benchmark=True,
)
prices = prices.loc[:"2022-08-31", ASSET_COLS]
real_rets = pct_returns(prices)

pools, pool_path, pool_kind = load_regime_pools(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Returns:", real_rets.shape, real_rets.index[0].date(), "->", real_rets.index[-1].date())
print("Pool source:", pool_kind, pool_path)
for k, windows in pools.items():
    print(f"  regime {k}: {windows.shape}")


## Rolling windows

Non-overlapping-step rolling MVO (`lookback=252`, `invest=60`, `step=21`). Windows are labeled calm or crisis with `classify_regime`. Single-seed plots use seed 42; the same run stores all `SEEDS` for the multi-seed section.


In [ ]:
results_ms = rolling_experiment(
    real_rets, pools, mix_grid=MIX_GRID, lookback=LOOKBACK, invest=INVEST,
    step=STEP, seeds=SEEDS, random_state=SEED,
)
results = results_ms.loc[results_ms["seed"] == SEED].copy()
print(results.shape)
display(results.groupby(["regime", "n_synth_series"])[["sharpe", "calmar", "ann_mu", "ann_sd"]].mean().round(3))
display(results.groupby(["regime", "n_synth_series"])[["sharpe", "calmar"]].median().round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_metric_curve(results, metric="sharpe", estimator="mean", spy_col="spy_sharpe",
                  title="Sharpe vs N specialist series", ax=axes[0])
plot_metric_curve(results, metric="calmar", estimator="median", spy_col="spy_calmar",
                  title="Calmar vs N specialist series", ax=axes[1])
plt.tight_layout()
plt.show()

display(paired_deltas(results))
display(results.groupby(["regime", "n_synth_series"])["max_weight_used"].mean().round(4))


## Shock tests

Train on unshocked lookback windows, then apply each shock to the 300-day test window. Median across rolling start dates.


In [ ]:
SHOCK_LOOKBACK, SHOCK_INVEST = 252, 300
shock = shock_experiment(
    real_rets, pools, mix_grid=MIX_GRID, lookback=SHOCK_LOOKBACK,
    invest=SHOCK_INVEST, step=STEP, seed=SEED,
)
shock_med = (
    shock.groupby(["bucket", "n_synth_series"])[["sharpe", "calmar", "CVaR_5%", "max_drawdown"]]
    .median()
    .reset_index()
)
metrics = ["sharpe", "calmar", "CVaR_5%", "max_drawdown"]
comp = shock_med.loc[shock_med["n_synth_series"] == 0, ["bucket"]].copy()
for n in MIX_GRID:
    sub = shock_med.loc[shock_med["n_synth_series"] == n, ["bucket"] + metrics]
    comp = comp.merge(sub.rename(columns={m: f"{m}_n{n}" for m in metrics}), on="bucket")
display(comp.round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for scenario, grp in shock_med.groupby("bucket"):
    grp = grp.sort_values("n_synth_series")
    axes[0].plot(grp["n_synth_series"], grp["max_drawdown"], marker="o", label=scenario)
    axes[1].plot(grp["n_synth_series"], grp["CVaR_5%"], marker="o", label=scenario)
for ax, ylabel in zip(axes, ["Max drawdown (median)", "CVaR 5% (median)"]):
    ax.set_xlabel("N specialist series")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=7)
axes[0].set_title("Shock max drawdown")
axes[1].set_title("Shock CVaR")
plt.tight_layout()
plt.show()


## Long-short

Same rolling windows as above, with short positions allowed.


In [ ]:
ls_results = rolling_experiment(
    real_rets, pools, mix_grid=MIX_GRID, lookback=LOOKBACK, invest=INVEST,
    step=STEP, seeds=(SEED,), allow_short=True, random_state=SEED,
)
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
colors = {"calm": "steelblue", "crisis": "tomato"}
for col, (metric, est, ylabel) in enumerate([
    ("sharpe", "mean", "Mean Sharpe"),
    ("calmar", "median", "Median Calmar"),
]):
    lo = results.groupby(["regime", "n_synth_series"])[metric].agg(est).reset_index()
    ls = ls_results.groupby(["regime", "n_synth_series"])[metric].agg(est).reset_index()
    for regime in ("calm", "crisis"):
        c = colors[regime]
        axes[0, col].plot(lo.loc[lo.regime == regime, "n_synth_series"],
                          lo.loc[lo.regime == regime, metric], marker="o", color=c, label=regime)
        axes[1, col].plot(ls.loc[ls.regime == regime, "n_synth_series"],
                          ls.loc[ls.regime == regime, metric], marker="o", color=c, label=regime)
    axes[0, col].set_title(f"Long-only {ylabel}")
    axes[1, col].set_title(f"Long-short {ylabel}")
    axes[1, col].set_xlabel("N specialist series")
    for ax in (axes[0, col], axes[1, col]):
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Multi-seed

Uncertainty from specialist-path sampling. Sharpe is the mean across windows and seeds; Calmar is the mean of per-seed medians.


In [ ]:
colors = {"calm": "steelblue", "crisis": "tomato"}
BAND_YLIM = (-2.0, 5.0)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, metric, ylabel in [
    (axes[0], "sharpe", "Mean Sharpe (+-1 SE)"),
    (axes[1], "calmar", "Median Calmar (+-1 SE across seeds)"),
]:
    if metric == "sharpe":
        agg = results_ms.groupby(["regime", "n_synth_series"])[metric].agg(["mean", "sem"]).reset_index()
    else:
        per = results_ms.groupby(["seed", "regime", "n_synth_series"])[metric].median().reset_index()
        agg = per.groupby(["regime", "n_synth_series"])[metric].agg(["mean", "std", "count"]).reset_index()
        agg["sem"] = agg["std"] / np.sqrt(agg["count"])
    display(agg.round(4))
    for regime, grp in agg.groupby("regime"):
        grp = grp.sort_values("n_synth_series")
        ax.plot(grp["n_synth_series"], grp["mean"], marker="o", color=colors[regime], label=regime)
        ax.fill_between(grp["n_synth_series"], grp["mean"] - grp["sem"], grp["mean"] + grp["sem"],
                        alpha=0.15, color=colors[regime])
    ax.axhline(0, color="black", lw=0.6, alpha=0.35)
    ax.set_ylim(*BAND_YLIM)
    ax.set_xlabel("N specialist series")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
axes[0].set_title("Sharpe")
axes[1].set_title("Calmar")
plt.tight_layout()
plt.show()


## Daily rebalance

HMM and weights refresh every 21 trading days; holdings are applied daily. Split A ends 2017; Split B ends 2019. Buy-and-hold is A001.


In [ ]:
n_train_a = int(real_rets.loc[:"2017-12-31"].shape[0])
n_train_b = int(real_rets.loc[:"2019-12-31"].shape[0])
print("Split A test:", real_rets.index[n_train_a].date(), "->", real_rets.index[-1].date())
print("Split B test:", real_rets.index[n_train_b].date(), "->", real_rets.index[-1].date())

daily_a = daily_paths(
    real_rets, pools, n_synth_list=N_SYNTH_LIST, lookback=LOOKBACK,
    test_start=n_train_a, hmm_step=HMM_STEP, seed=SEED,
)
daily_b = daily_paths(
    real_rets, pools, n_synth_list=N_SYNTH_LIST, lookback=LOOKBACK,
    test_start=n_train_b, hmm_step=HMM_STEP, seed=SEED,
)
bh_a = real_rets[ASSET_COLS[0]].loc[daily_a[0].index]
bh_b = real_rets[ASSET_COLS[0]].loc[daily_b[0].index]
d_colors = {0: "steelblue", 1: "darkred", 2: "purple", 5: "darkorange", 20: "seagreen"}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, paths, bh, title in [
    (axes[0], daily_a, bh_a, "Split A (2018-2022)"),
    (axes[1], daily_b, bh_b, "Split B (2020-2022)"),
]:
    for n in N_SYNTH_LIST:
        ax.plot((1 + paths[n]).cumprod(), label=f"n_synth={n}", color=d_colors[n], lw=1.6)
    ax.plot((1 + bh).cumprod(), label=f"B&H {ASSET_COLS[0]}", color="gray", ls="--", lw=1.3)
    ax.axhline(1.0, color="black", ls=":", lw=0.8, alpha=0.5)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)
axes[0].set_ylabel("Cumulative wealth")
plt.tight_layout()
plt.show()
print("Split A")
display(path_metrics(daily_a, bh_a, f"B&H {ASSET_COLS[0]}").round(3))
print("Split B")
display(path_metrics(daily_b, bh_b, f"B&H {ASSET_COLS[0]}").round(3))
wealth_b = {n: (1 + daily_b[n]).cumprod() for n in N_SYNTH_LIST}
wealth_bh_b = (1 + bh_b).cumprod()


In [ ]:
daily_full = daily_paths(
    real_rets, pools, n_synth_list=N_SYNTH_LIST, lookback=LOOKBACK,
    test_start=LOOKBACK, hmm_step=HMM_STEP, seed=SEED,
)
bh_full = real_rets[ASSET_COLS[0]].loc[daily_full[0].index]
wealth_full = {n: (1 + daily_full[n]).cumprod() for n in N_SYNTH_LIST}

fig, ax = plt.subplots(figsize=(13, 5))
for n in N_SYNTH_LIST:
    ax.plot(wealth_full[n], label=f"n_synth={n}", color=d_colors[n], lw=1.8)
ax.plot((1 + bh_full).cumprod(), label=f"B&H {ASSET_COLS[0]}", color="gray", ls="--", lw=1.4)
ax.axhline(1.0, color="black", ls=":", lw=0.8, alpha=0.5)
ax.set_title("Daily MVO wealth, full sample after lookback")
ax.set_ylabel("Cumulative wealth")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
display(path_metrics(daily_full, bh_full, f"B&H {ASSET_COLS[0]}").round(3))


## Constrained MVO

Per-asset cap of 25%. Static weights are fit once at the Split B train/test cut; daily weights refresh every 21 days.


In [ ]:
static_b = static_paths(real_rets, pools, n_synth_list=N_SYNTH_LIST, test_start=n_train_b, seed=SEED)
cap_static = static_paths(
    real_rets, pools, n_synth_list=N_SYNTH_LIST, test_start=n_train_b,
    seed=SEED, max_weight=MAX_W,
)
cap_daily = daily_paths(
    real_rets, pools, n_synth_list=N_SYNTH_LIST, lookback=LOOKBACK,
    test_start=n_train_b, hmm_step=HMM_STEP, seed=SEED, max_weight=MAX_W,
)
idx = daily_b[0].index
fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=True)
for ax, n in zip(axes, N_SYNTH_LIST):
    ax.plot((1 + static_b[n].loc[idx]).cumprod(), color="steelblue", lw=1.4, label="Unc. static")
    ax.plot(wealth_b[n], color="steelblue", lw=1.4, ls="--", label="Unc. daily")
    ax.plot((1 + cap_static[n].loc[idx]).cumprod(), color="darkorange", lw=1.6, label="Cap static")
    ax.plot((1 + cap_daily[n]).cumprod(), color="darkorange", lw=1.6, ls="--", label="Cap daily")
    ax.plot(wealth_bh_b, color="gray", lw=1.0, ls=":", label=f"B&H {ASSET_COLS[0]}")
    ax.axhline(1.0, color="black", ls=":", lw=0.6, alpha=0.4)
    ax.set_title(f"n_synth={n}")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=6)
axes[0].set_ylabel("Cumulative wealth")
plt.suptitle(f"Unconstrained vs cap {MAX_W:.0%}, Split B")
plt.tight_layout()
plt.show()


Long-short with the same 25% cap (weights in [-0.25, 0.25]).


In [ ]:
ls_cap_static = static_paths(
    real_rets, pools, n_synth_list=N_SYNTH_LIST, test_start=n_train_b,
    seed=SEED, allow_short=True, max_weight=MAX_W,
)
ls_cap_daily = daily_paths(
    real_rets, pools, n_synth_list=N_SYNTH_LIST, lookback=LOOKBACK,
    test_start=n_train_b, hmm_step=HMM_STEP, seed=SEED,
    allow_short=True, max_weight=MAX_W,
)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, n in zip(axes, [0, 5, 20]):
    ax.plot((1 + cap_static[n].loc[idx]).cumprod(), color="steelblue", lw=1.6, label="LO static")
    ax.plot((1 + cap_daily[n]).cumprod(), color="steelblue", lw=1.6, ls="--", label="LO daily")
    ax.plot((1 + ls_cap_static[n].loc[idx]).cumprod(), color="firebrick", lw=1.6, label="LS static")
    ax.plot((1 + ls_cap_daily[n]).cumprod(), color="firebrick", lw=1.6, ls="--", label="LS daily")
    ax.plot(wealth_bh_b, color="gray", lw=1.0, ls=":", label=f"B&H {ASSET_COLS[0]}")
    ax.axhline(1.0, color="black", ls=":", lw=0.6, alpha=0.4)
    ax.set_title(f"n_synth={n}")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)
axes[0].set_ylabel("Cumulative wealth")
plt.suptitle(f"Long-only vs long-short, cap {MAX_W:.0%}, Split B")
plt.tight_layout()
plt.show()


## Correlation

MVO uses the correlation structure. Compare real calm/crisis blocks with a specialist panel made by concatenating one window from each regime.


In [ ]:
def corr_matrix(df):
    return df.corr().to_numpy()

calm = real_rets.loc["2014-01-01":"2018-09-30"]
covid = real_rets.loc["2020-02-01":"2020-06-01"]
rates = real_rets.loc["2022-01-01":"2022-08-31"]
all_rets = real_rets.loc["2014-01-01":"2022-08-31"]
synth_corr = specialist_corr_matrix(pools, rng_seed=SEED)

panels = [
    ("Calm (2014-2018)", corr_matrix(calm)),
    ("COVID (2020 Feb-Jun)", corr_matrix(covid)),
    ("Rates (2022)", corr_matrix(rates)),
    ("All (2014-2022)", corr_matrix(all_rets)),
    ("Specialist windows", synth_corr),
]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (title, C) in zip(axes, panels):
    im = ax.imshow(C, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_title(title, fontsize=9)
    ax.set_xticks(range(len(ASSET_COLS)))
    ax.set_xticklabels(ASSET_COLS, rotation=90, fontsize=7)
    ax.set_yticks(range(len(ASSET_COLS)))
    ax.set_yticklabels(ASSET_COLS, fontsize=7)
plt.colorbar(im, ax=axes[-1], fraction=0.046)
plt.tight_layout()
plt.show()
